# 가상문서 임베딩 


In [40]:

import logging 
from typing import List 
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient


In [41]:
COLLECTION_NAME = "invest_index"
QDRANT_URL = "http://localhost:6333"

In [42]:
def get_loader(file_path: str):
    try: 
        docs = []   
        loaders = [TextLoader(file_path)]
        for loader in loaders:
            docs.extend(loader.load())
        return docs 
    except FileExistsError as e:
        print(f"failed load file : {file_path}")
        return []


def get_recursive_splitter(chunk_size :int=1000, chunk_overlap: int=200):
    return RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)


def split_docs(docs):
    recursive_spliter = get_recursive_splitter()
    return recursive_spliter.split_documents(docs) 

def get_embedding():
    return OpenAIEmbeddings(
        model="bge-m3",
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
        check_embedding_ctx_length=False)


def initialize_vector_store(docs: List[Document]) -> QdrantVectorStore:
    split_documents = split_docs(docs)

    return QdrantVectorStore.from_documents(
        documents=split_documents,
        embedding=get_embedding(),
        url=QDRANT_URL,
        collection_name=COLLECTION_NAME,
    )

def create_vector_store() -> QdrantVectorStore:
    return QdrantVectorStore.from_existing_collection(
        embedding=get_embedding(),
        url=QDRANT_URL,
        collection_name=COLLECTION_NAME,
    )


def add_documents_to_vector_store(vector_store: QdrantVectorStore, docs: List[Document]) -> None:
    vector_store.add_documents(docs)



In [43]:
file_path = "./data/How_to_invest_money.txt" 
docs = get_loader(file_path)
vector_store = initialize_vector_store(docs)



